In [0]:
# Notebook 3: Construcción del modelo estrella (capa Gold)
# Lee la tabla limpia y produce dim_customer, dim_item, dim_date, fact_sales

df_clean = spark.table("workspace.default.retail_clean")

print(f"Filas en retail_clean: {df_clean.count()}")
df_clean.printSchema()

In [0]:
from pyspark.sql.functions import row_number, regexp_extract, lit
from pyspark.sql.window import Window

# Una fila por cada Customer ID único
dim_customer = (df_clean
                .select("customer_id")
                .distinct()
                .orderBy("customer_id"))

# Generar la llave surrogate (entero secuencial)
window = Window.orderBy("customer_id")
dim_customer = dim_customer.withColumn("customer_key", row_number().over(window))

# Extraer el código numérico (los dos dígitos finales)
dim_customer = dim_customer.withColumn(
    "customer_code",
    regexp_extract("customer_id", r"CUST_(\d+)", 1)
)

# Reordenar columnas
dim_customer = dim_customer.select("customer_key", "customer_id", "customer_code")

# Guardar como tabla Gold
(dim_customer.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")
 .saveAsTable("workspace.default.dim_customer"))

print(f"dim_customer creada con {dim_customer.count()} filas:")
dim_customer.show()

In [0]:
from pyspark.sql.functions import col, row_number, regexp_extract
from pyspark.sql.window import Window

# Una fila por cada item único (incluye categoría + código de categoría)
dim_item = (df_clean
            .select("item", "category")
            .distinct()
            .orderBy("item"))

window = Window.orderBy("item")
dim_item = dim_item.withColumn("item_key", row_number().over(window))

# Extraer el sufijo del item (ej: BEV, BUT, FOOD) como category_code
dim_item = dim_item.withColumn(
    "category_code",
    regexp_extract("item", r"Item_\d+_(\w+)", 1)
)

# Renombrar columnas para alinearse con el modelo
dim_item = dim_item.withColumnRenamed("item", "item_id")
dim_item = dim_item.withColumn("item_name", col("item_id"))

# Reordenar columnas
dim_item = dim_item.select("item_key", "item_id", "item_name", "category", "category_code")

(dim_item.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")
 .saveAsTable("workspace.default.dim_item"))

print(f"dim_item creada con {dim_item.count()} filas:")
dim_item.show(10)

In [0]:
from pyspark.sql.functions import (
    explode, sequence, to_date as f_to_date, year, quarter, month,
    dayofmonth, dayofweek, date_format, expr
)

# Generar todas las fechas entre 2022-01-01 y 2025-12-31
df_dates = spark.sql("""
    SELECT explode(sequence(to_date('2022-01-01'), to_date('2025-12-31'), interval 1 day)) AS full_date
""")

# Agregar las columnas derivadas de la fecha
dim_date = (df_dates
    .withColumn("date_key", date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", year("full_date"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("day_of_week", dayofweek("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn("is_weekend", expr("day_of_week IN (1, 7)"))
)

# Reordenar columnas
dim_date = dim_date.select(
    "date_key", "full_date", "year", "quarter", "month", "month_name",
    "day", "day_of_week", "day_name", "is_weekend"
)

(dim_date.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")
 .saveAsTable("workspace.default.dim_date"))

print(f"dim_date creada con {dim_date.count()} filas")
dim_date.show(5)

In [0]:
# Cargar las dimensiones recién creadas
dim_customer_df = spark.table("workspace.default.dim_customer")
dim_item_df = spark.table("workspace.default.dim_item")
dim_date_df = spark.table("workspace.default.dim_date")

# Construir la tabla de hechos con los joins
fact_sales = (df_clean
    # Join con dim_customer
    .join(dim_customer_df.select("customer_key", "customer_id"), on="customer_id", how="inner")
    # Join con dim_item (la columna se llama "item" en clean, "item_id" en dim)
    .join(dim_item_df.select("item_key", "item_id"),
          df_clean["item"] == dim_item_df["item_id"], "inner")
    # Join con dim_date
    .join(dim_date_df.select("date_key", "full_date"),
          df_clean["transaction_date"] == dim_date_df["full_date"], "inner")
    # Castear quantity a INT (estaba como double)
    .withColumn("quantity", col("quantity").cast("int"))
    # Seleccionar solo las columnas finales según el modelo
    .select(
        "transaction_id",
        "customer_key",
        "item_key",
        "date_key",
        "quantity",
        "price_per_unit",
        "total_spent",
        "payment_method",
        "location",
        "discount_applied"
    )
)

(fact_sales.write
 .mode("overwrite")
 .option("delta.columnMapping.mode", "name")
 .saveAsTable("workspace.default.fact_sales"))

print(f"fact_sales creada con {fact_sales.count()} filas")
fact_sales.show(5)

In [0]:
# Verificar integridad del modelo estrella
print("=== RESUMEN DEL MODELO ESTRELLA ===\n")
print(f"fact_sales:   {spark.table('workspace.default.fact_sales').count()} filas")
print(f"dim_customer: {spark.table('workspace.default.dim_customer').count()} filas")
print(f"dim_item:     {spark.table('workspace.default.dim_item').count()} filas")
print(f"dim_date:     {spark.table('workspace.default.dim_date').count()} filas")

# Ejemplo de query OLAP: ventas totales por categoría
print("\n=== TOP 5 CATEGORÍAS POR INGRESOS ===")
spark.sql("""
    SELECT i.category, ROUND(SUM(f.total_spent), 2) AS total_ingresos
    FROM workspace.default.fact_sales f
    JOIN workspace.default.dim_item i ON f.item_key = i.item_key
    GROUP BY i.category
    ORDER BY total_ingresos DESC
    LIMIT 5
""").show()